In [ ]:
# Principal:    ivan
# Role:         data_engineer
# Catalog Role: engineer_catalog_role
# Привилегии:   CATALOG_MANAGE_CONTENT, CATALOG_MANAGE_ACCESS (catalog-уровень)
#
# Матрица доступов:
#   bronze: полный (LIST + READ + METADATA + WRITE + CREATE + DROP)
#   silver: полный (LIST + READ + METADATA + WRITE + CREATE + DROP)
#   gold:   полный (LIST + READ + METADATA + WRITE + CREATE + DROP)
#
# FORBIDDEN: нет RBAC-ограничений, проверяем обработку ошибок Spark

In [ ]:
import os
from pyspark.sql import SparkSession

client_id = os.environ["IVAN_CLIENT_ID"]
client_secret = os.environ["IVAN_CLIENT_SECRET"]
credential = f"{client_id}:{client_secret}"

spark = SparkSession.builder \
    .appName("lakehouse-ivan-rbac") \
    .config("spark.sql.catalog.lakehouse.credential", credential) \
    .getOrCreate()

spark

In [ ]:
spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("SHOW TABLES IN lakehouse.bronze").show(truncate=False)
spark.sql("SHOW TABLES IN lakehouse.silver").show(truncate=False)
spark.sql("SHOW TABLES IN lakehouse.gold").show(truncate=False)

In [ ]:
print("[ALLOWED] TABLE_LIST — lakehouse.bronze")
spark.sql("SHOW TABLES IN lakehouse.bronze").show(truncate=False)

In [ ]:
print("[ALLOWED] TABLE_READ_DATA — lakehouse.bronze.raw_categories")
spark.sql("""
    SELECT
        id,
        name,
        description
    FROM lakehouse.bronze.raw_categories
    LIMIT 3
""").show(truncate=False)

In [ ]:
print("[ALLOWED] TABLE_FULL_METADATA — lakehouse.bronze.raw_orders")
spark.sql("DESCRIBE TABLE EXTENDED lakehouse.bronze.raw_orders").show(truncate=False)

In [ ]:
print("[ALLOWED] TABLE_WRITE_DATA — INSERT в lakehouse.bronze.raw_orders")
spark.sql("""
    INSERT INTO lakehouse.bronze.raw_orders
    VALUES (99999, 1, 'completed', CAST(100.00 AS DECIMAL(12,2)), '2026-01-01')
""")
print("INSERT ok")

spark.sql("""
    SELECT
        id,
        customer_id,
        status,
        total_amount,
        order_date
    FROM lakehouse.bronze.raw_orders
    WHERE id = 99999
""").show(truncate=False)

spark.sql("DELETE FROM lakehouse.bronze.raw_orders WHERE id = 99999")
print("DELETE ok")

In [ ]:
print("[ALLOWED] TABLE_CREATE — lakehouse.bronze._test_bronze_probe")
try:
    spark.sql("DROP TABLE IF EXISTS lakehouse.bronze._test_bronze_probe")
except Exception:
    pass
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.bronze._test_bronze_probe (
        id   INT,
        name STRING
    ) USING iceberg
""")
print("CREATE ok")

In [ ]:
spark.range(100) \
    .selectExpr("CAST(id AS INT) AS id", "CONCAT('test_', CAST(id AS STRING)) AS name") \
    .write.mode("append").saveAsTable("lakehouse.bronze._test_bronze_probe")

cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.bronze._test_bronze_probe
""").collect()[0]["cnt"]
print(f"Загружено строк: {cnt}")

spark.sql("""
    SELECT
        id,
        name
    FROM lakehouse.bronze._test_bronze_probe
    LIMIT 3
""").show(truncate=False)

In [ ]:
try:
    spark.sql("DROP TABLE IF EXISTS lakehouse.bronze._test_bronze_probe")
    print("DROP ok")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"DROP пропущен (нет прав): {short}")

In [ ]:
print("[ALLOWED] TABLE_LIST — lakehouse.silver")
spark.sql("SHOW TABLES IN lakehouse.silver").show(truncate=False)

In [ ]:
print("[ALLOWED] TABLE_READ_DATA — lakehouse.silver.customers")
spark.sql("""
    SELECT
        id,
        name,
        email,
        city,
        created_at
    FROM lakehouse.silver.customers
    LIMIT 3
""").show(truncate=False)

In [ ]:
print("[ALLOWED] TABLE_FULL_METADATA — lakehouse.silver.orders")
spark.sql("DESCRIBE TABLE EXTENDED lakehouse.silver.orders").show(truncate=False)

In [ ]:
print("[ALLOWED] TABLE_WRITE_DATA — INSERT в lakehouse.silver.orders")
spark.sql("""
    INSERT INTO lakehouse.silver.orders
    VALUES (99998, 1, 'completed', CAST(99.99 AS DECIMAL(12,2)), CAST('2026-01-01' AS DATE))
""")
print("INSERT ok")

spark.sql("""
    SELECT
        id,
        customer_id,
        status,
        total_amount,
        order_date
    FROM lakehouse.silver.orders
    WHERE id = 99998
""").show(truncate=False)

spark.sql("DELETE FROM lakehouse.silver.orders WHERE id = 99998")
print("DELETE ok")

In [ ]:
print("[ALLOWED] TABLE_CREATE — lakehouse.silver._test_silver_probe")
try:
    spark.sql("DROP TABLE IF EXISTS lakehouse.silver._test_silver_probe")
except Exception:
    pass
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.silver._test_silver_probe (
        id   INT,
        name STRING
    ) USING iceberg
""")
print("CREATE ok")

In [ ]:
spark.range(100) \
    .selectExpr("CAST(id AS INT) AS id", "CONCAT('test_', CAST(id AS STRING)) AS name") \
    .write.mode("append").saveAsTable("lakehouse.silver._test_silver_probe")

cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.silver._test_silver_probe
""").collect()[0]["cnt"]
print(f"Загружено строк: {cnt}")

spark.sql("""
    SELECT
        id,
        name
    FROM lakehouse.silver._test_silver_probe
    LIMIT 3
""").show(truncate=False)

In [ ]:
try:
    spark.sql("DROP TABLE IF EXISTS lakehouse.silver._test_silver_probe")
    print("DROP ok")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"DROP пропущен (нет прав): {short}")

In [ ]:
print("[ALLOWED] TABLE_LIST — lakehouse.gold")
spark.sql("SHOW TABLES IN lakehouse.gold").show(truncate=False)

In [ ]:
print("[ALLOWED] TABLE_READ_DATA — lakehouse.gold.mart_sales_by_category")
spark.sql("""
    SELECT
        category_name,
        month,
        total_revenue,
        order_count,
        avg_check
    FROM lakehouse.gold.mart_sales_by_category
    LIMIT 3
""").show(truncate=False)

In [ ]:
print("[ALLOWED] TABLE_FULL_METADATA — lakehouse.gold.mart_sales_by_category")
spark.sql("DESCRIBE TABLE EXTENDED lakehouse.gold.mart_sales_by_category").show(truncate=False)

In [ ]:
print("[ALLOWED] TABLE_WRITE_DATA — INSERT в lakehouse.gold.mart_top_customers")
spark.sql("""
    INSERT INTO lakehouse.gold.mart_top_customers
    VALUES (
        99999,
        'test_customer',
        1,
        CAST(1 AS BIGINT),
        CAST(100.00 AS DECIMAL(14,2)),
        'Low'
    )
""")
print("INSERT ok")

spark.sql("""
    SELECT
        customer_id,
        name,
        recency_days,
        frequency,
        monetary,
        segment
    FROM lakehouse.gold.mart_top_customers
    WHERE customer_id = 99999
""").show(truncate=False)

spark.sql("DELETE FROM lakehouse.gold.mart_top_customers WHERE customer_id = 99999")
print("DELETE ok")

In [ ]:
print("[ALLOWED] TABLE_CREATE — lakehouse.gold._test_gold_probe")
try:
    spark.sql("DROP TABLE IF EXISTS lakehouse.gold._test_gold_probe")
except Exception:
    pass
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.gold._test_gold_probe (
        id   INT,
        name STRING
    ) USING iceberg
""")
print("CREATE ok")

In [ ]:
spark.range(100) \
    .selectExpr("CAST(id AS INT) AS id", "CONCAT('test_', CAST(id AS STRING)) AS name") \
    .write.mode("append").saveAsTable("lakehouse.gold._test_gold_probe")

cnt = spark.sql("""
    SELECT
        COUNT(*) AS cnt
    FROM lakehouse.gold._test_gold_probe
""").collect()[0]["cnt"]
print(f"Загружено строк: {cnt}")

spark.sql("""
    SELECT
        id,
        name
    FROM lakehouse.gold._test_gold_probe
    LIMIT 3
""").show(truncate=False)

In [ ]:
try:
    spark.sql("DROP TABLE IF EXISTS lakehouse.gold._test_gold_probe")
    print("DROP ok")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"DROP пропущен (нет прав): {short}")

In [ ]:
# Не RBAC — проверка обработки ошибок несуществующих объектов
print("[FORBIDDEN] Несуществующий namespace lakehouse.archive")
try:
    spark.sql("SHOW TABLES IN lakehouse.archive").show(truncate=False)
    print("[ПРОВАЛ] Доступ разрешён — ожидалась ошибка")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Ошибка: {short}")
    print()

print("[FORBIDDEN] Несуществующая таблица lakehouse.bronze.nonexistent")
try:
    spark.sql("""
        SELECT
            id
        FROM lakehouse.bronze.nonexistent
        LIMIT 1
    """).show(truncate=False)
    print("[ПРОВАЛ] Доступ разрешён — ожидалась ошибка")
except Exception as e:
    short = "\n".join(str(e).split("\n")[:3])
    print(f"[ОЖИДАЕМО] Ошибка: {short}")
    print()